# The Richards Equation
_1-D water flow in unsaturated soil_  
  
Caoimhe Mannion, November 2026  
  
**References**  
Jury, W. A., Gardner, W. R., & Gardner, W. H. (1991). _Soil Physics_ (5th ed.). Wiley.  
Tindall, J. A., & Kunkel, J. R. (1999). _Unsaturated Zone Hydrology for Scientists and Engineers_. Prentice Hall.  
Bouwer, H. (1978). _Groundwater Hydrology_. McGraw-Hill.  
Brooks, R. H., & Corey, A. T. (1964). Hydraulic Properties of Porous Media. _Hydrology Papers Colorado State University._

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import math
from scipy.interpolate import RectBivariateSpline as rbs
import sympy as sp

In [1]:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake

--2026-11-24 15:43:04--  https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh
Resolving fem-on-colab.github.io (fem-on-colab.github.io)... 185.199.109.153, 185.199.111.153, 185.199.108.153, ...
Connecting to fem-on-colab.github.io (fem-on-colab.github.io)|185.199.109.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4868 (4.8K) [application/x-sh]
Saving to: ‘/tmp/firedrake-install.sh’

/tmp/firedrake-inst 100%[===================>]   4.75K  --.-KB/s    in 0s      

2026-11-24 15:43:04 (46.7 MB/s) - ‘/tmp/firedrake-install.sh’ saved [4868/4868]

+ INSTALL_PREFIX=/usr/local
++ echo /usr/local
++ awk -F/ '{print NF-1}'
+ INSTALL_PREFIX_DEPTH=2
+ PROJECT_NAME=fem-on-colab
+ SHARE_PREFIX=/usr/local/share/fem-on-colab
+ FIREDRAKE_INSTALLED=/usr/local/share/fem-on-colab/firedrake.installed
+ [[ ! -f /usr/local/share/fem-on-colab/firedrake.installed ]]
+ PYBIND11_INSTALL_SCRIPT_PATH=https://github.com/fem-on-colab/fem-on-colab.github.io/raw/cd97

In [2]:
from firedrake import *

In [3]:
!pip install IRKsome

In [4]:
from irksome import GaussLegendre, Dt, TimeStepper


### Parameters

In [6]:
# Values for Touchet Silt Loam, Brooks & Corey 1964
porosity = 0.485 #fraction
h_b = 75 #cm, bubbling pressure, expressed as head
k_perm = 0.6 # intrinsic permeability of the soil, sq microns
lambda_ = 1.82 # pore size distribution index
S_r = 0.270 # residual saturation (vwc of pore space)

In [7]:
# Get required parameters
k_perm = k_perm*10**(-8) #unit conversion to sq cm
rho = 1 # density of water, g/cm^3
g = 9.81 # acceleration due to gravity m/s
mu = 0.01002 # viscosity of water at 20C, g/cm*s
K_s = k_perm*rho*g/mu
print(f"K_s = {K_s:.3e}")

theta_r = S_r*porosity
theta_s = porosity
print(f"theta_r, theta_s = {theta_r:.3f}, {theta_s:.3f}")

K_s = 5.874e-06
theta_r, theta_s = 0.131, 0.485


### Conditions
$\theta(t, z)$

In [8]:
# initial condition
#theta_t0 = some function
T = 300 #s e.g.
t_range = np.linspace(0,T,T+1)

# boundary conditions: edges of range
theta_z0 = theta_r
theta_zd = theta_s
d = 100 #cm, depth in soil
z_range = np.linspace(0,d,d+1)

## Functions
$\theta$ is volumetric water content: volume of water per unit volume of bulk soil. Dimensionless: units $m^{3}/m^{3}$, given as percentage by volume    
$K\left(\theta \right)$ is hydraulic conductivity as a function of volumetric water content $\theta$  
$K_{s}$ is hydraulic conductivity at saturation, a constant  
$S_{e}$ is effective saturation, a function in $\theta$  
$λ$ is the pore size distribution index, a constant  
${\theta }_{r}$ is the residual or irreducible water content (physical minimum)  
${\theta }_{s}$ is the volume of water content at saturation, which is the value of porosity, unless entrapped air reduces its value (physical maximum)  
${\psi }_{m}$ is the matric potential: energy change due to the soil matrix’s attraction for water.   
${\psi }_{b}$ is the air-entry/bubbling pressure, assumed to be the matric potential at which the largest pore in the soil begins to drain  
Potential is expressed as energy per unit weight: $J/N=\frac{kgm^{2}s^{-2}}{kgms^{-2}}=m$. Typical for Richards Equation: expressed as head $h$ with units length, equal to the height of a static column of fluid whose pressure at the base is equal to the static pressure at that point.  

### Brooks-Corey Retention Curve  
$$ S_e = \frac{\theta-\theta_r}{\theta_s-\theta_r} = \left(\frac{\psi_b}{\psi_m}\right)^{\lambda} = \left(\frac{h_b}{|h|}\right)^{\lambda} $$
For $\psi_b > \psi_m$ and $\psi_b, \psi_m$ same units and sign  
$h_b > h$ and $h_b, h$ same units (length). $h$ is negative in the unsaturated zone, $h_b$ is a positive constant, a characteristic of the soil.
  
If $\psi_b \leq \psi_m$ or $h_b \leq h$, $S_e = 1$
  
### Hydraulic Conductivity
$$ K\left(\theta \right)=K_{s}{\left(S_{e}\right)}^{3+2/\lambda} $$
$$ K\left(\theta \right)=K_{s}{\left(\frac{\theta-\theta_r}{\theta_s-\theta_r}\right)}^{3+2/\lambda} $$

  
### Water Diffusivity Function
$$ D_{w}\left(\theta \right)=K\left(\theta \right)\frac{dh}{d\theta }$$
$$D_{w}\left(\theta \right)=K\left(\theta \right)\frac{h_{b}}{\lambda \left({\theta }_{s}-{\theta }_{r}\right)}S_{e}^{-1/\lambda -1}$$

### Water Content Form of Richards Equation
$$\frac{\partial \theta}{\partial t}=\frac{\partial}{\partial z}\left[D_w(\theta)\left(\frac{\partial \theta}{\partial z}\right)+\frac{\partial K(\theta)}{\partial z}\right] $$
Where:  
$\theta$ is volumetric water content.  
$t$ is time  
$z$ is depth in soil (0 at surface, positive with increasing depth)  
$D_{w} \left(\theta \right)$ is the water diffusivity function  
$K\left(\theta \right)$ is hydraulic conductivity

In [9]:
### Water Content Form ###

def K(theta):
    """Burdine Hydraulic Conductivity"""
    return K_s*S_e(theta)**(3+2/lambda_)

def S_e(theta):
    """Brooks-Corey Retention Curve"""
    return (theta-theta_r)/(theta_s-theta_r)

def D(theta):
    """Water Diffusivity Function"""
    return K(theta)*h_b/(lambda_(theta_s-theta_r))*S_e(theta)**(-1/lambda_ - 1)